<a href="https://colab.research.google.com/github/Dima200206/-2/blob/main/%D0%9B%D0%913_%D0%92%D0%BB%D0%B0%D1%81%D0%B5%D0%BD%D0%BA%D0%BE%20%D0%94%D0%BC%D0%B8%D1%82%D1%80%D0%BE%20%D0%86%D0%BD%D1%82%D0%B5%D0%BB%D0%B5%D0%BA%D1%82%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D1%96%20%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8%20%D0%A4%D0%86%D0%A2%201-4%20%D0%BC%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Лабораторна 3

In [1]:
# -----------------------------
# БАЗА ПРАВИЛ (ЛБ1)
# -----------------------------
rules = [

 {"name": "R1",
  "conditions": {"genre_preference": "comedy", "mood": "happy"},
  "conclusion": ("movie_type", "light_comedy")},

 {"name": "R2",
  "conditions": {"genre_preference": "action", "intensity_preference": "high"},
  "conclusion": ("movie_type", "action_thriller")},

 {"name": "R3",
  "conditions": {"genre_preference": "drama", "mood": "sad"},
  "conclusion": ("movie_type", "emotional_drama")},

 {"name": "R4",
  "conditions": {"genre_preference": "fantasy", "available_time": "long"},
  "conclusion": ("movie_type", "epic_fantasy")},

 {"name": "R5",
  "conditions": {"watch_with_family": "yes", "genre_preference": "comedy"},
  "conclusion": ("movie_type", "family_comedy")},

 {"name": "R6",
  "conditions": {"watch_with_family": "yes", "intensity_preference": "low"},
  "conclusion": ("movie_type", "family_movie")}
]

# -----------------------------
# ЗВОРОТНИЙ ЛАНЦЮЖОК (з input)
# -----------------------------
def backward_chain(goal_attr, goal_value, facts, depth=0):
    indent = "  " * depth
    log = []

    # 1. Якщо факт вже є
    if facts.get(goal_attr) == goal_value:
        return True, [f"{indent}✔ Факт знайдено: {goal_attr}={goal_value}"]

    # 2. Шукаємо правила
    for rule in rules:
        res_attr, res_val = rule["conclusion"]

        if res_attr == goal_attr and res_val == goal_value:
            log.append(f"{indent}➡ Перевіряємо правило {rule['name']}")

            all_true = True
            sub_log = []

            # перевіряємо умови
            for attr, val in rule["conditions"].items():
                success, inner_log = backward_chain(attr, val, facts, depth+1)
                sub_log.extend(inner_log)

                if not success:
                    all_true = False
                    break

            if all_true:
                facts[goal_attr] = goal_value
                log.extend(sub_log)
                log.append(f"{indent} {rule['name']} спрацювало")
                return True, log

    # 3.  ІНТЕРАКТИВНИЙ ЗАПИТ (ГОЛОВНЕ ЗАВДАННЯ 3)
    if goal_attr not in facts:
        answer = input(f"Введіть значення для {goal_attr}: ")

        facts[goal_attr] = answer

        if answer == goal_value:
            return True, [f"{indent} Користувач підтвердив {goal_attr}={goal_value}"]
        else:
            return False, [f"{indent} Користувач ввів {goal_attr}={answer}"]

    return False, [f"{indent} Не вдалося довести {goal_attr}={goal_value}"]

In [4]:
facts = {
    "genre_preference": "action"
}

goal = ("movie_type", "action_thriller")

result, log = backward_chain(goal[0], goal[1], facts)

print("\nЖУРНАЛ:")
for l in log:
    print(l)

print("\nРЕЗУЛЬТАТ:", result)
print("ФАКТИ:", facts)

Введіть значення для intensity_preference: high

ЖУРНАЛ:
➡ Перевіряємо правило R2
  ✔ Факт знайдено: genre_preference=action
   Користувач підтвердив intensity_preference=high
 R2 спрацювало

РЕЗУЛЬТАТ: True
ФАКТИ: {'genre_preference': 'action', 'intensity_preference': 'high', 'movie_type': 'action_thriller'}


In [5]:
facts = {"genre_preference": "action"}

In [6]:
# -----------------------------
# BACKWARD CHAIN + LOG
# -----------------------------
def backward_chain(goal_attr, goal_value, facts, depth=0, visited=None):
    if visited is None:
        visited = set()

    indent = "  " * depth
    log = []

    # 1. Перевірка факту
    if facts.get(goal_attr) == goal_value:
        return True, [f"{indent}✔ Факт: {goal_attr}={goal_value} вже відомий"]

    # 2. Пошук правил
    found_rule = False

    for rule in rules:
        res_attr, res_val = rule["conclusion"]

        if res_attr == goal_attr and res_val == goal_value:
            found_rule = True

            if rule["name"] in visited:
                continue

            log.append(f"{indent} Гіпотеза: використати {rule['name']}")

            visited.add(rule["name"])

            all_true = True
            sub_log = []

            # перевірка умов
            for attr, val in rule["conditions"].items():
                success, inner_log = backward_chain(attr, val, facts, depth+1, visited)
                sub_log.extend(inner_log)

                if not success:
                    all_true = False
                    log.append(f"{indent} {rule['name']} НЕ спрацювало (умова {attr}={val} не виконана)")
                    break

            if all_true:
                facts[goal_attr] = goal_value
                log.extend(sub_log)
                log.append(f"{indent} {rule['name']} СПРАЦЮВАЛО → {goal_attr}={goal_value}")
                return True, log

    # 3. Інтерактивний запит
    if goal_attr not in facts:
        answer = input(f"Введіть значення для {goal_attr}: ")
        facts[goal_attr] = answer

        if answer == goal_value:
            return True, [f"{indent} Користувач підтвердив {goal_attr}={goal_value}"]
        else:
            return False, [f"{indent} Користувач ввів {goal_attr}={answer}"]

    # 4. Якщо нічого не знайдено
    if not found_rule:
        return False, [f"{indent} Немає правил для {goal_attr}"]

    return False, [f"{indent} Не вдалося довести {goal_attr}={goal_value}"]

In [7]:
def explain(goal, facts):
    print("\n" + "="*40)
    print(f"ПЕРЕВІРКА: {goal[0]} = {goal[1]}")

    success, log = backward_chain(goal[0], goal[1], facts)

    print("\nЖУРНАЛ ДОВЕДЕННЯ:")
    for step in log:
        print(step)

    print("\nРЕЗУЛЬТАТ:")
    if success:
        print(" Гіпотеза доведена")
    else:
        print(" Гіпотеза НЕ доведена")

    print("\nКІНЦЕВІ ФАКТИ:")
    print(facts)

In [8]:
facts = {
    "genre_preference": "action"
}

goal = ("movie_type", "action_thriller")

explain(goal, facts)


ПЕРЕВІРКА: movie_type = action_thriller
Введіть значення для intensity_preference: action
Введіть значення для movie_type: high

ЖУРНАЛ ДОВЕДЕННЯ:
 Користувач ввів movie_type=high

РЕЗУЛЬТАТ:
 Гіпотеза НЕ доведена

КІНЦЕВІ ФАКТИ:
{'genre_preference': 'action', 'intensity_preference': 'action', 'movie_type': 'high'}


In [10]:
# -----------------------------
# СЦЕНАРІЇ ТЕСТУВАННЯ
# -----------------------------

scenarios = [

    {
        "name": "Сценарій 1 — Успішне доведення",
        "facts": {
            "genre_preference": "action",
            "intensity_preference": "high"
        },
        "goal": ("movie_type", "action_thriller")
    },

    {
        "name": "Сценарій 2 — Інший успішний випадок",
        "facts": {
            "genre_preference": "comedy",
            "mood": "happy"
        },
        "goal": ("movie_type", "light_comedy")
    },

    {
        "name": "Сценарій 3 — Недостатньо даних (буде input)",
        "facts": {
            "genre_preference": "action"
        },
        "goal": ("movie_type", "action_thriller")
    },

    {
        "name": "Сценарій 4 — Неможливо довести",
        "facts": {
            "genre_preference": "fantasy",
            "mood": "happy"
        },
        "goal": ("movie_type", "action_thriller")
    },

    {
        "name": "Сценарій 5 — Конфлікт / невідповідність",
        "facts": {
            "genre_preference": "action",
            "intensity_preference": "low"
        },
        "goal": ("movie_type", "action_thriller")
    }
]

# -----------------------------
# ЗАПУСК ВСІХ СЦЕНАРІЇВ
# -----------------------------

for scenario in scenarios:
    print("\n" + "="*50)
    print(scenario["name"])

    facts = dict(scenario["facts"])  # копія
    goal = scenario["goal"]

    explain(goal, facts)


Сценарій 1 — Успішне доведення

ПЕРЕВІРКА: movie_type = action_thriller

ЖУРНАЛ ДОВЕДЕННЯ:
 Гіпотеза: використати R2
  ✔ Факт: genre_preference=action вже відомий
  ✔ Факт: intensity_preference=high вже відомий
 R2 СПРАЦЮВАЛО → movie_type=action_thriller

РЕЗУЛЬТАТ:
 Гіпотеза доведена

КІНЦЕВІ ФАКТИ:
{'genre_preference': 'action', 'intensity_preference': 'high', 'movie_type': 'action_thriller'}

Сценарій 2 — Інший успішний випадок

ПЕРЕВІРКА: movie_type = light_comedy

ЖУРНАЛ ДОВЕДЕННЯ:
 Гіпотеза: використати R1
  ✔ Факт: genre_preference=comedy вже відомий
  ✔ Факт: mood=happy вже відомий
 R1 СПРАЦЮВАЛО → movie_type=light_comedy

РЕЗУЛЬТАТ:
 Гіпотеза доведена

КІНЦЕВІ ФАКТИ:
{'genre_preference': 'comedy', 'mood': 'happy', 'movie_type': 'light_comedy'}

Сценарій 3 — Недостатньо даних (буде input)

ПЕРЕВІРКА: movie_type = action_thriller
Введіть значення для intensity_preference: happy
Введіть значення для movie_type: comedy

ЖУРНАЛ ДОВЕДЕННЯ:
 Користувач ввів movie_type=comedy

РЕЗУЛЬ

In [11]:
import networkx as nx
import matplotlib.pyplot as plt

def draw_proof_tree(facts, log):
    G = nx.DiGraph()

    # додаємо факти
    for key, value in facts.items():
        G.add_node(f"{key}={value}", color='lightblue')

    # аналіз логів
    for entry in log:
        if "СПРАЦЮВАЛО" in entry:
            parts = entry.split(" ")
            rule_name = parts[1]

            G.add_node(rule_name, color='lightgreen')

        elif "Гіпотеза" in entry:
            rule_name = entry.split(" ")[-1]
            G.add_node(rule_name, color='yellow')

    # зв’язки (простий варіант)
    for key, value in facts.items():
        for node in G.nodes:
            if "R" in node:
                G.add_edge(f"{key}={value}", node)

    colors = [G.nodes[n].get('color', 'gray') for n in G.nodes]

    pos = nx.spring_layout(G)
    nx.draw(G, pos, with_labels=True, node_color=colors, node_size=2000, font_size=8)

    plt.title("Схема доведення")
    plt.show()

In [12]:
def explain(goal, facts):
    print("\n" + "="*40)
    print(f"ПЕРЕВІРКА: {goal[0]} = {goal[1]}")

    success, log = backward_chain(goal[0], goal[1], facts)

    print("\nЖУРНАЛ:")
    for step in log:
        print(step)

    print("\nРЕЗУЛЬТАТ:")
    print(" Доведено" if success else " Не доведено")

    # додаємо граф
    draw_proof_tree(facts, log)